In [ ]:
# =====================================================================
# PART 1: DATA IMPORT AND INITIAL EXPLORATION
# =====================================================================
print("--- PART 1: IMPORTATION ET EXPLORATION INITIALE ---")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras import models, layers, optimizers

# Fixer les graines pour la reproductibilité
np.random.seed(42)
tf.random.set_seed(42)

# Téléchargement et extraction automatique du dataset s'il n'est pas présent
dataset_url = "https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip"
zip_path = "household_power_consumption.zip"

if not os.path.exists('household_power_consumption.txt'):
    print("Téléchargement du dataset Household Power Consumption...")
    tf.keras.utils.get_file(zip_path, dataset_url, extract=True, cache_dir='.', cache_subdir='')
    if os.path.exists('individual_household_electric_power_consumption.txt'):
        os.rename('individual_household_electric_power_consumption.txt', 'household_power_consumption.txt')

# Chargement optimisé du jeu de données
# Le dataset utilise ';' comme séparateur et '?' pour les valeurs manquantes
print("Chargement des données dans un DataFrame Pandas...")
df = pd.read_csv('household_power_consumption.txt', sep=';', 
                 parse_dates={'Datetime': ['Date', 'Time']}, 
                 infer_datetime_format=True, 
                 low_memory=False, 
                 na_values=['?'], 
                 index_col='Datetime')

print(f"\n✅ Forme du dataset (Shape) : {df.shape}")
print("\nAperçu des premières lignes :")
print(df.head())
print("\nTypes des colonnes :")
print(df.dtypes)


# =====================================================================
# PART 2: HANDLING MISSING VALUES
# =====================================================================
print("\n--- PART 2: GESTION DES VALEURS MANQUANTES ---")

print("Valeurs manquantes détectées par colonne avant traitement :")
print(df.isnull().sum())

# Remplacement des valeurs manquantes par la moyenne de chaque colonne respective
for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mean())

print(f"\n✅ Vérification finale : {df.isnull().sum().sum()} valeur(s) manquante(s) restante(s).")


# =====================================================================
# PART 3: DATA VISUALIZATION
# =====================================================================
print("\n--- PART 3: VISUALISATION DES DONNÉES (RESAMPLING) ---")

# 1. Rééchantillonnage de 'Global_active_power' à la journée ('D') - Somme et Moyenne
gap_daily_sum = df['Global_active_power'].resample('D').sum()
gap_daily_mean = df['Global_active_power'].resample('D').mean()

plt.figure(figsize=(14, 5))
plt.plot(gap_daily_sum.index, gap_daily_sum, label='Somme Journalière', color='royalblue', alpha=0.7)
plt.plot(gap_daily_mean.index, gap_daily_mean * 100, label='Moyenne Journalière (Échelle x100)', color='darkorange', alpha=0.9)
plt.title('Global Active Power : Évolution de la Somme vs Moyenne par Jour')
plt.xlabel('Date')
plt.ylabel('Valeur')
plt.legend()
plt.tight_layout()
plt.show()

# 2. Rééchantillonnage de 'Global_intensity' à la journée ('D') - Moyenne et Écart-type
gi_daily_mean = df['Global_intensity'].resample('D').mean()
gi_daily_std = df['Global_intensity'].resample('D').std()

plt.figure(figsize=(14, 5))
plt.plot(gi_daily_mean.index, gi_daily_mean, label='Moyenne Journalière', color='crimson')
plt.fill_between(gi_daily_mean.index, gi_daily_mean - gi_daily_std, gi_daily_mean + gi_daily_std, 
                 color='crimson', alpha=0.15, label='Écart-type ($\sigma$)')
plt.title('Global Intensity Journalière : Moyenne et Dispersion')
plt.xlabel('Date')
plt.ylabel('Intensité')
plt.legend()
plt.tight_layout()
plt.show()


# =====================================================================
# PART 4: DATA PREPROCESSING FOR LSTM
# =====================================================================
print("\n--- PART 4: PRÉPRÉPARATION DES DONNÉES POUR LE LSTM ---")

# Afin d'éviter des calculs trop lourds dans Colab (données à la minute),
# nous rééchantillonnons à l'heure ('H') pour la variable 'Global_active_power'.
df_hourly = df['Global_active_power'].resample('H').mean().to_frame()

# 1. Normalisation [0, 1]
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(df_hourly)

# 2. Split en ensembles d'entraînement (80%) et de test (20%)
train_size = int(len(scaled_data) * 0.8)
train_data, test_data = scaled_data[:train_size], scaled_data[train_size:]

# 3. Fonction pour structurer les données en fenêtres glissantes (Séquences temporelles)
def create_sequences(data, time_steps=24):
    X, y = [], []
    for i in range(len(data) - time_steps):
        X.append(data[i:(i + time_steps), 0])
        y.append(data[i + time_steps, 0])
    return np.array(X), np.array(y)

# On utilise les 24 heures précédentes pour prédire la 25ème
TIME_STEPS = 24
X_train, y_train = create_sequences(train_data, TIME_STEPS)
X_test, y_test = create_sequences(test_data, TIME_STEPS)

# Reshape au format requis par les réseaux LSTM : [Samples, Time Steps, Features]
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))

print(f"Dimensions finales pour l'entraînement : X_train {X_train.shape} | y_train {y_train.shape}")


# =====================================================================
# PART 5: BUILDING AN LSTM MODEL
# =====================================================================
print("\n--- PART 5: CONFIGURATION DE L'ARCHITECTURE LSTM ---")

# Définition du modèle
model = models.Sequential([
    layers.Input(shape=(X_train.shape[1], 1)),
    layers.LSTM(units=50, return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(units=50, return_sequences=False),
    layers.Dropout(0.2),
    layers.Dense(units=1) # Sortie de régression linéaire pour la valeur continue suivante
])

model.summary()

# Compilation
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='mean_squared_error'
)


# =====================================================================
# PART 6: TRAINING AND EVALUATING THE LSTM MODEL
# =====================================================================
print("\n--- PART 6: ENTRAÎNEMENT ET ÉVALUATION DU MODÈLE ---")

# Vérification de la disponibilité du GPU avant l'entraînement
print("Dispositif matériel détecté :", tf.config.list_physical_devices('GPU'))

# Lancement de l'apprentissage
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_test, y_test),
    verbose=1
)

# 1. Tracé de l'évolution des pertes (Loss curves)
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Perte Entraînement (Train Loss)', color='blue')
plt.plot(history.history['val_loss'], label='Perte Validation (Val Loss)', color='orange')
plt.title('Évolution de la Perte (MSE) pendant l\'Entraînement')
plt.xlabel('Epochs')
plt.ylabel('Mean Squared Error')
plt.legend()
plt.show()

# 2. Évaluation visuelle sur les données de Test (Zoom sur les 150 premières heures)
predictions = model.predict(X_test, verbose=0)
predictions_inverse = scaler.inverse_transform(predictions)
y_test_inverse = scaler.inverse_transform(y_test.reshape(-1, 1))

plt.figure(figsize=(14, 5))
plt.plot(y_test_inverse[:150], label='Valeurs Réelles', color='black', alpha=0.8)
plt.plot(predictions_inverse[:150], label='Prédictions LSTM', color='dodgerblue', linestyle='--')
plt.title('Comparaison : Valeurs Réelles vs Prédictions du LSTM (Zoom 150h)')
plt.xlabel('Temps (Heures)')
plt.ylabel('Global Active Power (kW)')
plt.legend()
plt.tight_layout()
plt.show()

print("\n✅ Script exécuté avec succès. Tu n'as plus qu'à pousser ton notebook sur GitHub !")